In [5]:
import easyocr
import pickle

reader = easyocr.Reader(['en'])

model = pickle.load(open("receipt_model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

result = reader.readtext("sample_receipt.jpg")

food_keywords = [
    "burger",
    "whopper",
    "fry",
    "drink",
    "pizza",
    "pepperoni",
    "combo",
    "sandwich",
    "coffee",
    "water"
]

for r in result:

    text = r[1].strip()

    if not text:
        continue

    text_lower = text.lower()

    if not any(keyword in text_lower for keyword in food_keywords):
        continue

    # Rule-based override
    if any(keyword in text_lower for keyword in food_keywords):
        print(f"{text} -> Food")
        continue

    vector = vectorizer.transform([text])

    prediction = model.predict(vector)

    print(f"{text} -> {prediction[0]}")
    

/opt/anaconda3/envs/reciept-ml/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Burger K -> Food
Regular Combo -> Food
Combo Marker -> Food
WHOPPER CHS -> Food
SM FRY -> Food
SM SOFT  DRINK -> Food


In [8]:
import easyocr
import pickle

reader = easyocr.Reader(['en'])

model = pickle.load(open("receipt_model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

result = reader.readtext("sample_receipt.jpg")
ignore_words = [
    "phone",
    "tax",
    "total",
    "subtotal",
    "survey",
    "visa",
    "auth",
    "host",
    "order",
    "check",
    "code",
    "payment",
    "balance"
]

for r in result:

    text = r[1].strip()

    if not text:
        continue

    text_lower = text.lower()

    # Skip short text
    if len(text) < 4:
        continue

    # Skip long numeric strings
    if any(char.isdigit() for char in text) and len(text) > 6:
        continue

    # Skip common receipt metadata
    if any(word in text_lower for word in ignore_words):
        continue

    vector = vectorizer.transform([text])

    # Get confidence
    confidence = model.predict_proba(vector).max()

    # Skip low-confidence predictions
    if confidence < 0.70:
        continue

    prediction = model.predict(vector)

    print(f"{text} -> {prediction[0]} ({confidence:.2f})")



/opt/anaconda3/envs/reciept-ml/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Burger K -> Food (0.84)


In [9]:
sample = ["Burger King WHOPPER"]

vector = vectorizer.transform(sample)

prediction = model.predict(vector)

print(prediction[0])

Food


In [10]:
samples = [
    "Burger King WHOPPER",
    "Burger King FRY",
    "Burger King SOFT DRINK"
]

vectors = vectorizer.transform(samples)

predictions = model.predict(vectors)

for text, pred in zip(samples, predictions):
    print(f"{text} -> {pred}")

Burger King WHOPPER -> Food
Burger King FRY -> Food
Burger King SOFT DRINK -> Food


In [11]:
samples = [
    "Burger King WHOPPER",
    "Burger King FRY",
    "Burger King SOFT DRINK"
]

vectors = vectorizer.transform(samples)

predictions = model.predict(vectors)

probabilities = model.predict_proba(vectors)

for text, pred, probs in zip(samples, predictions, probabilities):
    confidence = max(probs)
    print(f"{text} -> {pred} ({confidence:.2f})")

Burger King WHOPPER -> Food (0.83)
Burger King FRY -> Food (0.81)
Burger King SOFT DRINK -> Food (0.74)
